In [ ]:
import os
import sys
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/Stocks'
except ImportError:
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from IPython.display import HTML, display
from portfolio.signals import build_signal_table, ASSET_META

# =========================================================================
# 1. FETCH ALL SIGNALS
# =========================================================================
results = build_signal_table()

# =========================================================================
# 2. BUILD HTML TABLE
# =========================================================================

def buy_color(signal):
    if signal == "BUY NOW": return "#1B5E20"
    if signal == "BUY DIP": return "#E65100"
    return "#B71C1C"

def sell_color(action):
    if "SELL NOW" in action: return "#B71C1C"
    if "NEAR TARGET" in action: return "#E65100"
    if "HOLD FOREVER" in action: return "#1B5E20"
    return "#1a1a1a"

def trend_color(trend):
    if trend == "UPTREND": return "#1B5E20"
    if trend == "DOWNTREND": return "#B71C1C"
    return "#E65100"

def basket_bg(basket):
    colors = {
        "Core ETF": "#E3F2FD",
        "Nuclear": "#FFF8DC",
        "Quantum": "#F3E6F5",
        "Quantum (paused)": "#F3E6F5",
        "Cyber": "#FFEBEE",
        "Industrial": "#E8EAF6",
        "SpecGrowth": "#E0F7FA",
    }
    return colors.get(basket, "#FFFFFF")

html = []

# --- CSS ---
html.append("""<style>
.sig-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 12px; }
.sig-table th { background: #2C3E50; color: white; padding: 8px 10px; text-align: left; font-weight: bold; white-space: nowrap; }
.sig-table td { padding: 6px 10px; border-bottom: 1px solid #ddd; color: #1a1a1a; white-space: nowrap; }
.sig-table tr:hover { filter: brightness(0.95); }
.sig-header { font-size: 18px; font-weight: bold; margin-bottom: 8px; color: #2C3E50; }
.sig-sub { font-size: 12px; color: #555; margin-bottom: 16px; }
.sig-legend { font-size: 11px; color: #555; margin-top: 12px; line-height: 1.8; }
</style>""")

html.append('<div class="sig-header">Portfolio Buy / Sell Signal Dashboard</div>')
html.append('<div class="sig-sub">Signals based on RSI-14, 50/200-SMA trend, 52-week range, and asset-specific catalysts.</div>')

# --- Table ---
html.append('<table class="sig-table">')
html.append('<tr>')
html.append('<th>Ticker</th><th>Name</th><th>Basket</th><th>Shares</th>')
html.append('<th>Price</th><th>RSI</th><th>Trend</th><th>From 52w High</th>')
html.append('<th>Buy Signal</th><th>Buy Reason</th>')
html.append('<th>Sell Action</th><th>Sell Detail</th>')
html.append('<th>Catalyst / Thesis</th>')
html.append('</tr>')

# Group by basket order
basket_order = ["Core ETF", "Nuclear", "Quantum", "Quantum (paused)", "Cyber", "Industrial", "SpecGrowth"]
sorted_results = sorted(results, key=lambda r: (
    basket_order.index(r["basket"]) if r["basket"] in basket_order else 99,
    r["ticker"]
))

for r in sorted_results:
    bg = basket_bg(r["basket"])
    bc = buy_color(r["buy_signal"])
    sc = sell_color(r["sell_action"])
    tc = trend_color(r["trend"])

    target_str = f' → ${r["sell_target"]:.0f}' if r["sell_target"] else ""

    html.append(f'<tr style="background:{bg};">')
    html.append(f'<td><b>{r["ticker"]}</b></td>')
    html.append(f'<td>{r["name"]}</td>')
    html.append(f'<td>{r["basket"]}</td>')
    html.append(f'<td>{r["shares"]}</td>')
    html.append(f'<td>${r["price"]:.2f}{target_str}</td>')
    html.append(f'<td>{r["rsi"]:.0f}</td>')
    html.append(f'<td style="color:{tc}; font-weight:bold;">{r["trend"]}</td>')
    html.append(f'<td>{r["pct_from_high"]:.1f}%</td>')
    html.append(f'<td style="color:{bc}; font-weight:bold;">{r["buy_signal"]}</td>')
    html.append(f'<td>{r["buy_reason"]}</td>')
    html.append(f'<td style="color:{sc}; font-weight:bold;">{r["sell_action"]}</td>')
    html.append(f'<td>{r["sell_detail"]}</td>')
    html.append(f'<td style="max-width:250px; white-space:normal;">{r["catalyst"]}</td>')
    html.append('</tr>')

html.append('</table>')

# --- Legend ---
html.append('<div class="sig-legend">')
html.append('<b>Buy Signals:</b> ')
html.append('<span style="color:#1B5E20; font-weight:bold;">BUY NOW</span> = strong entry (oversold/support) | ')
html.append('<span style="color:#E65100; font-weight:bold;">BUY DIP</span> = scale in on weakness | ')
html.append('<span style="color:#B71C1C; font-weight:bold;">WAIT</span> = overbought or downtrend')
html.append('<br><b>Sell Actions:</b> ')
html.append('<span style="color:#1B5E20; font-weight:bold;">HOLD FOREVER</span> = core ETF, DCA only | ')
html.append('<b>TARGET $X</b> = sell at price target | ')
html.append('<b>SELL @ EVENT</b> = sell on catalyst date | ')
html.append('<span style="color:#E65100; font-weight:bold;">NEAR TARGET</span> = within 10%, tighten stop | ')
html.append('<span style="color:#B71C1C; font-weight:bold;">SELL NOW</span> = at/above target')
html.append('<br><b>Strategy:</b> hold_forever = never sell | accumulate = buy dips, sell at target | catalyst = binary event | swing = momentum trade')
html.append('<br>⚠️ This is NOT financial advice. Signals are mechanical indicators, not recommendations.')
html.append('</div>')

# =========================================================================
# 3. RENDER
# =========================================================================
display(HTML('\n'.join(html)))
print('\nSignal dashboard rendered.')

